# Random Embedding + Frozen CNN Baseline

The embedding layer is randomly initialized and kept frozen (`trainable=False`).

In [1]:
import numpy as np
import pandas as pd
import keras
from keras import layers
from sklearn.metrics import accuracy_score, f1_score

import sys
sys.path.append("..")
from utils.preprocessing import LABELS, encode_labels, fit_tokenizer, encode_texts

keras.utils.set_random_seed(42)

In [2]:
train = pd.read_csv("../data/train.csv")
val = pd.read_csv("../data/val.csv")

tokenizer = fit_tokenizer(train.sentence)
X_train = encode_texts(tokenizer, train.sentence)
X_val = encode_texts(tokenizer, val.sentence)
y_train = encode_labels(train.label)
y_val = encode_labels(val.label)
vocab_size = len(tokenizer.word_index) + 1

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Embedding(vocab_size, 100, trainable=False),
    layers.Conv1D(64, 3, activation="relu"),
    layers.GlobalMaxPooling1D(),
    layers.Dense(len(LABELS), activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    verbose=1,
)

Epoch 1/10
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.5932 - loss: 0.9396 - val_accuracy: 0.5936 - val_loss: 0.9090
Epoch 2/10
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5950 - loss: 0.8863 - val_accuracy: 0.6277 - val_loss: 0.8750
Epoch 3/10
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6401 - loss: 0.8441 - val_accuracy: 0.6505 - val_loss: 0.8462
Epoch 4/10
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6543 - loss: 0.8096 - val_accuracy: 0.6598 - val_loss: 0.8231
Epoch 5/10
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6660 - loss: 0.7785 - val_accuracy: 0.6732 - val_loss: 0.8022
Epoch 6/10
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6818 - loss: 0.7481 - val_accuracy: 0.6825 - val_loss: 0.7835
Epoch 7/10
87/91 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6976 - loss: 0.7238

In [5]:
# Print statistics and add them to metrics.csv

predictions = model.predict(X_val, verbose=0).argmax(axis=1)
class_f1 = f1_score(y_val, predictions, average=None)

row = {
    "condition": "baseline",
    "embedding_config": "random_frozen_100d",
    "architecture": "cnn_64",
    "accuracy_overall": accuracy_score(y_val, predictions),
    "f1_overall": f1_score(y_val, predictions, average="macro"),
    "f1_negative": class_f1[0],
    "f1_neutral": class_f1[1],
    "f1_positive": class_f1[2],
}

print("Random frozen embedding CNN statistics")
print(f"Accuracy: {row['accuracy_overall']:.4f}")
print(f"Macro-F1: {row['f1_overall']:.4f}")
print(f"Negative F1: {row['f1_negative']:.4f}")
print(f"Neutral F1: {row['f1_neutral']:.4f}")
print(f"Positive F1: {row['f1_positive']:.4f}")

Random frozen embedding CNN statistics
Accuracy: 0.7104
Macro-F1: 0.5799
Negative F1: 0.4198
Neutral F1: 0.8120
Positive F1: 0.5080


In [6]:
metrics = pd.read_csv("../results/metrics.csv")
last_baseline = metrics[metrics["condition"] == "baseline"].index.max()
metrics = pd.concat(
    [
        metrics.iloc[:last_baseline + 1],
        pd.DataFrame([row]),
        metrics.iloc[last_baseline + 1:],
    ],
    ignore_index=True,
)
metrics.to_csv("../results/metrics.csv", index=False)
metrics

,condition,embedding_config,architecture,accuracy_overall,f1_overall,accuracy_hard,f1_hard,accuracy_nonhard,f1_nonhard,f1_negative,f1_neutral,f1_positive
0,baseline,tfidf_1-2gram,logreg,0.776629,0.717241,NaN,NaN,NaN,NaN,0.631579,0.845771,0.674374
1,baseline,random_init_100d,lstm_64_dropout,0.698035,0.464626,NaN,NaN,NaN,NaN,0.000000,0.832653,0.561224
2,baseline,tfidf_1-2gram,logreg,0.776629,0.717241,NaN,NaN,NaN,NaN,0.631579,0.845771,0.674374
3,baseline,random_init_100d,lstm_64_dropout,0.698035,0.464626,NaN,NaN,NaN,NaN,0.000000,0.832653,0.561224
4,baseline,random_frozen_100d,cnn_64,0.710445,0.579916,NaN,NaN,NaN,NaN,0.419753,0.811985,0.508009
5,random_control,finbert_finetuned,cnn_64_finbert,0.779731,0.710225,0.538776,0.466455,0.806904,0.743418,0.597771,0.847918,0.684985
6,random_control,finbert_frozen,cnn_64_finbert,0.798759,0.749278,0.618367,0.455547,0.819102,0.771809,0.683070,0.854425,0.710340
7,targeted,finbert_finetuned,cnn_64_finbert,0.778697,0.709929,0.552577,0.477068,0.803908,0.741177,0.601528,0.847326,0.680935
8,targeted,finbert_frozen,cnn_64_finbert,0.798345,0.749777,0.624742,0.451130,0.817701,0.770973,0.686379,0.853857,0.709095
9,targeted,finbert_pretrained,transformer_finbert,0.857438,0.838963,0.582474,0.560134,NaN,NaN,NaN,NaN,NaN
